In [16]:
import requests
import pandas as pd

API_KEY = "0400e243bc36696930dd82f2d8cdd8a3d79b4ac8"

variables = {
    "B19013_001E": "median_income",
    "B01003_001E": "population",
    "B01002_001E": "median_age",
    "B15003_001E": "pop_25plus",
    "B15003_022E": "bachelors",
    "B15003_023E": "masters",
    "B15003_024E": "professional",
    "B15003_025E": "doctorate",
}

response = requests.get(
    "https://api.census.gov/data/2022/acs/acs5",
    params={
        "get": ",".join(variables.keys()),
        "for": "zip code tabulation area:*",
        "key": API_KEY,
    },
)
data = response.json()
census = pd.DataFrame(data[1:], columns=data[0])

# Clean up
census = census.rename(columns=variables)
census = census.rename(columns={"zip code tabulation area": "zip"})

for col in variables.values():
    census[col] = pd.to_numeric(census[col], errors="coerce")
    census.loc[census[col] < 0, col] = float("nan")

census["pct_college"] = (
    census[["bachelors", "masters", "professional", "doctorate"]].sum(axis=1)
    / census["pop_25plus"]
)

census.head()

,median_income,population,median_age,pop_25plus,bachelors,masters,professional,doctorate,zip,pct_college
0,17526.0,16834.0,44.0,12295.0,1773.0,237.0,23.0,23.0,00601,0.167222
1,20260.0,37642.0,45.2,28321.0,4767.0,1181.0,69.0,109.0,00602,0.216306
2,17703.0,49075.0,45.0,36307.0,6001.0,2132.0,414.0,337.0,00603,0.244691
3,19603.0,5590.0,46.2,4118.0,287.0,129.0,0.0,0.0,00606,0.101020
4,22796.0,25542.0,44.4,19085.0,3199.0,846.0,151.0,17.0,00610,0.220749


In [17]:
census = census.rename(columns=variables)
census = census.rename(columns={"zip code tabulation area": "zip"})

# Convert to numbers and replace Census missing-value codes
for col in variables.values():
    census[col] = pd.to_numeric(census[col], errors="coerce")
    census.loc[census[col] < 0, col] = float("nan")

# Share of adults with a college degree
census["pct_college"] = (
    census[["bachelors", "masters", "professional", "doctorate"]].sum(axis=1)
    / census["pop_25plus"]
)

In [18]:
gaz = pd.read_csv("/Users/mayadoitch/Desktop/2026_Gaz_zcta_national.txt",
                  sep="|", dtype={"GEOID": str})
gaz.columns = gaz.columns.str.strip()

gaz = gaz[["GEOID", "ALAND_SQMI"]].rename(columns={"GEOID": "zip"})

census = census.merge(gaz, on="zip", how="left")
census["pop_density"] = census["population"] / census["ALAND_SQMI"]

In [19]:
census_out = census[["zip", "median_income", "population", "median_age",
                     "pct_college", "ALAND_SQMI", "pop_density"]].copy()

prefix = census_out["zip"].str[:3].astype(int)
census_out["state"] = None
census_out.loc[prefix.between(150, 196), "state"] = "PA"
census_out.loc[prefix.between(320, 349), "state"] = "FL"

census_out = census_out[census_out["state"].notna()]

census_out.to_csv("census_fl_pa.csv", index=False)
print(census_out["state"].value_counts())

state
PA    1833
FL    1013
Name: count, dtype: int64


In [21]:
census_out = census_out[census_out["state"] == "FL"]
census_out.to_csv("census_fl.csv", index=False)